# 03 — Gold: Analytics Table & KPIs

**Tickets:** A-01, A-02, A-03, A-04  
**Purpose:** Build the Gold analytical table and compute KPIs: revenue per zone per hour, average trip duration by day-of-week.

---

## Setup

In [0]:
import importlib
import src.constants
import src.transforms

importlib.reload(src.constants)
importlib.reload(src.transforms)

from src.constants import GOLD_FACT_TABLE, SILVER_TABLE  # noqa: E402
from src.transforms import (  # noqa: E402
    build_dim_location,
    build_dim_time,
    build_fact_trips,
)

print("Setup complete")

## Read Silver table

In [0]:
silver_df = spark.read.table(SILVER_TABLE)
print(f"Silver table: {SILVER_TABLE}")
print(f"Row count: {silver_df.count():,}")
print(f"Columns: {len(silver_df.columns)}")

## A-01 — Gold schema design

### Star schema overview

```
                  ┌─────────────────────┐
                  │   dim_location    │
                  │  (view, ~3–5K)    │
                  │─────────────────────│
                  │ zone_id   (PK)    │
                  │ zone_lat          │
                  │ zone_lon          │
                  └──────────┬──────────┘
                             │
┌───────────────────────────────────────────────┐
│              fact_trips (Delta)               │
│   Grain: pickup_zone × hour_of_day × day_of_week  │
│───────────────────────────────────────────────│
│ pickup_zone             (string, FK)          │
│ hour_of_day             (int, FK)             │
│ day_of_week             (int, FK)             │
│ is_weekend              (boolean, degenerate)  │
│ trip_count              (long)                 │
│ total_revenue           (double)               │
│ avg_fare_amount         (double)               │
│ avg_trip_distance       (double)               │
│ avg_trip_duration_min   (double)               │
│ avg_tip_amount          (double)               │
│ total_passengers        (long)                 │
└───────────────────────┬───────────────────────┘
                            │
                  ┌──────────┴──────────┐
                  │     dim_time       │
                  │  (view, 168 rows)  │
                  │─────────────────────│
                  │ hour_of_day  (PK)  │
                  │ day_of_week  (PK)  │
                  │ day_name           │
                  │ is_weekend         │
                  │ time_period        │
                  └─────────────────────┘
```

### Design decisions & rationale

| # | Decision | Rationale |
|---|----------|-----------|
| A | **Grain: pickup_zone × hour_of_day × day_of_week** | Directly serves A-03 (revenue/zone/hour), A-04 (duration/day), A-05 (demand heatmap), BQ-1 (demand by location/time), BQ-2 (fare drivers). |
| B | **Pickup zone only** (no dropoff in grain) | Dropoff is not needed by any downstream ticket (A-03, A-04, BQ-1, BQ-2). Including it would explode the table to zones² × 24 × 7 rows. |
| C | **Fact table = persisted Delta; dims = views** | The fact table aggregates 140M Silver rows to \~500K — persisting avoids expensive re-aggregation on every dashboard load. Dimension tables are tiny (168 rows for dim_time, \~3–5K for dim_location) and compute instantly, so views guarantee freshness with zero staleness risk. |
| D | **Exclude NULL pickup_zone** from fact table | \~1.6M Silver rows have NULLed GPS (Finding #10, cleaned in I-03). These cannot contribute to location-based analytics (BQ-1, A-03, A-05). |
| E | **is_weekend as degenerate dimension** | Deterministically derived from day_of_week (1=Sun, 7=Sat). Included in fact row for query convenience; not part of the PK. |
| F | **Transform functions in `src/transforms.py`** | Follows the established project pattern. Constants (DAY_NAME_MAP, TIME_PERIOD_BINS) in `src/constants.py`. |

### Known limitations from EDA findings

| Finding | Impact on Gold | Mitigation |
|---------|---------------|------------|
| #2 — Non-contiguous date range (Jan 2015 + Jan–Mar 2016, 9-month gap) | The grain collapses all months into `day_of_week` — no date/month dimension. A-03 revenue and A-04 duration averages are composites across 4 non-contiguous months. Seasonal patterns and year-over-year comparisons are **not possible**. | Acceptable for this dataset. A-06 dashboard narrative must acknowledge that all metrics represent a composite of Jan 2015 + Jan–Mar 2016, not a continuous period. |
| #13 — Cash tip blindspot (34% of trips) | `avg_tip_amount` in the fact table aggregates across all payment types. Cash trips have `tip_amount = 0` by design (tips not recorded, not absent), systematically deflating the metric. | A-03/A-04 do not use tip metrics, so core KPIs are unaffected. If dashboards display avg tip, they must caveat that it includes cash trips. BQ-4 tip models correctly use Silver filtered to `payment_type = 1`. |

### Downstream ticket coverage

| Ticket | What it reads from Gold | How |
|--------|------------------------|-----|
| A-02 | `fact_trips` | Build step — writes the fact table |
| A-03 | `fact_trips` | `GROUP BY pickup_zone, hour_of_day` → `SUM(total_revenue)` |
| A-04 | `fact_trips` JOIN `dim_time` | `GROUP BY day_name` → weighted avg of `avg_trip_duration_min` |
| A-05 | `fact_trips` JOIN `dim_location` | Heatmap: zone_lat/zone_lon × hour_of_day × trip_count |
| BQ-1 | `fact_trips` JOIN `dim_time` + `dim_location` | Demand = trip_count by zone, hour, day |
| BQ-2 | `fact_trips` | Revenue drivers: total_revenue vs avg_trip_distance, hour_of_day, pickup_zone |

## A-02 — Build Gold fact table

In [0]:
# A-02a — Build fact table (persisted as Delta)
fact_trips_df = build_fact_trips(silver_df)

# A-02b — Build dimension views (not persisted — computed on read)
dim_location_df = build_dim_location(silver_df)
dim_time_df = build_dim_time(spark)

# Summary stats
fact_count = fact_trips_df.count()
print(f"fact_trips rows:    {fact_count:,}")
print(f"dim_location rows:  {dim_location_df.count():,}")
print(f"dim_time rows:      {dim_time_df.count():,}")

print("\n=== fact_trips sample ===")
display(fact_trips_df.orderBy("pickup_zone", "hour_of_day", "day_of_week").limit(10))

print("\n=== dim_location sample ===")
display(dim_location_df.limit(5))

print("\n=== dim_time sample ===")
display(dim_time_df.orderBy("day_of_week", "hour_of_day").limit(10))

## A-03 — KPI 1: Revenue per zone per hour

In [0]:
# TODO: compute and display revenue per zone per hour

## A-04 — KPI 2: Average trip duration by day-of-week

In [0]:
# TODO: compute and display avg trip duration by day-of-week

## Write Gold Delta table

In [0]:
# Write fact_trips as a managed Delta table (dims are views, not persisted)
fact_trips_df.write.format("delta").mode("overwrite").saveAsTable(GOLD_FACT_TABLE)

row_count = spark.read.table(GOLD_FACT_TABLE).count()
print(f"Gold fact table written to: {GOLD_FACT_TABLE}")
print(f"Persisted rows: {row_count:,}")